In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
# Load Dataset

customer_support = pd.read_csv('/content/customer_support_text_classification.csv')

# **Task 1 Dataset Understanding**

In [35]:
# Number of records

print("Number of record: ", len(customer_support))

Number of record:  1500


In [36]:
# Target labels/classes

print(customer_support['sentiment_label'].unique())

['neutral' 'positive' 'negative']


In [37]:
# Sample text records

print(customer_support['customer_message'].head())

0    I need information about the payment process. ...
1        I need information about the payment process.
2    The refund process was fast and convenient. I ...
3    My refund is still pending and this experience...
4     Please tell me how to update my account details.
Name: customer_message, dtype: object


In [38]:
# Average text length

customer_support['text_length'] = customer_support['customer_message'].apply(lambda x: len(x.split()))
print("Average text length: " + str(customer_support['text_length'].mean()))

Average text length: 12.722666666666667


In [39]:
# Class distribution

print(customer_support['sentiment_label'].value_counts())

sentiment_label
neutral     524
negative    497
positive    479
Name: count, dtype: int64


# **Task 2 Text Preprocessing**

In [40]:
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [41]:
# Lowercasing

customer_support['customer_message'] = customer_support['customer_message'].str.lower()

In [42]:
import re
# Removing unnecessary symbols or special characters

customer_support['clean_text'] = customer_support['customer_message'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))

In [43]:
# Tokenization

nltk.download('punkt_tab')
customer_support['tokens'] = customer_support['clean_text'].apply(lambda x: word_tokenize(x))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [44]:
# Removing stopwords, if required

stop_words = set(stopwords.words('english'))
customer_support['tokens'] = customer_support['tokens'].apply(lambda x: [word for word in x if word not in stop_words])

customer_support['preprocessed_text'] = customer_support['tokens'].apply(lambda x: " ".join(x))

In [45]:
# Convert text to sequence

tokenizer = Tokenizer()
tokenizer.fit_on_texts(customer_support['preprocessed_text'])
sequences = tokenizer.texts_to_sequences(customer_support['preprocessed_text'])

In [46]:
# Padding Sequences

max_length = 20

padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

print("Original Text: ")
print(customer_support['customer_message'][0])

print("Padded Sequence: ")
print(padded_sequences[0])

print("Preprocessed Sequence: ")
print(customer_support['preprocessed_text'][0])


Original Text: 
i need information about the payment process. my ticket number is 78732. please respond as soon as possible.
Padded Sequence: 
[ 14 104  64  18   1   2 149   3   4   5   6   0   0   0   0   0   0   0
   0   0]
Preprocessed Sequence: 
need information payment process ticket number 78732 please respond soon possible


# **Task 3 Text Vectorization**


In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [48]:
from os import X_OK
# Text column

text = customer_support['customer_message']

# TF-IDF

tfidf = TfidfVectorizer()

# Convert text into vectors

X_tfidf = tfidf.fit_transform(text)

print("TF-IDF Shape:", X_tfidf.shape)

# Convert sparse matrix to array

print(X_tfidf.toarray())

TF-IDF Shape: (1500, 667)
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [49]:
# Tokenizer-based sequences

tokenizer = Tokenizer()

tokenizer.fit_on_texts(customer_support['preprocessed_text'])

sequences = tokenizer.texts_to_sequences(customer_support['preprocessed_text'])

padded_sequence = pad_sequences(sequences, maxlen = 20, padding = 'post', truncating = 'post')

print("Sample Sequence:")
print(sequences[0])

print("Padded Sequence: ")
print(padded_sequence)


Sample Sequence:
[14, 104, 64, 18, 1, 2, 149, 3, 4, 5, 6]
Padded Sequence: 
[[ 14 104  64 ...   0   0   0]
 [ 14 104  64 ...   0   0   0]
 [ 13  18  37 ...   0   0   0]
 ...
 [ 14  34  35 ...   0   0   0]
 [ 13  18  37 ...   0   0   0]
 [ 50  51  52 ...   0   0   0]]


# Explain why text must be converted into vectors before being used by a model.

Text must be converted to vectors because model only understand numbers.

Techniques like TF-IDF and tokenizer-based sequences transform text into machine-readable numerical representations for training NLP models.

# **Task 4 Baseline Model**


In [50]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [51]:
# Define features and target

X = customer_support['customer_message']
y = customer_support['sentiment_label']

In [52]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [53]:
# TF-IDF Vectorization

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [54]:
# Train Logistic Regression Model

model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

LogisticRegression()

In [55]:
# Prediction

y_pred = model.predict(X_test_tfidf)

In [56]:
# Evaluate the model

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 1.0


In [57]:
# Classification Report

print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       109
     neutral       1.00      1.00      1.00       104
    positive       1.00      1.00      1.00        87

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



In [58]:
# Confusion matrix

print(confusion_matrix(y_test, y_pred))

[[109   0   0]
 [  0 104   0]
 [  0   0  87]]


# **Task 5  Sequence Model or Conceptual Architecture**

In [59]:
from tensorflow.keras.utils import to_categorical
# Encode target labels
label_encoder = LabelEncoder()
y_encoded_all = label_encoder.fit_transform(y)
y_categorical_all = to_categorical(y_encoded_all)

In [60]:
# Build a model

model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=128, input_length=20))
model.add(LSTM(64))
model.add(Dense(3, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [61]:
# Compile

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [32]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Perform train-test split for the padded sequences and categorical labels
X_train_seq, X_test_seq, y_train_cat, y_test_cat = train_test_split(
    padded_sequence, y_categorical_all, test_size=0.2, random_state=42
)

# Train
model.fit(X_train_seq, y_train_cat, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.4948 - loss: 1.0201 - val_accuracy: 0.6667 - val_loss: 0.5561
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9427 - loss: 0.1663 - val_accuracy: 1.0000 - val_loss: 0.0116
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 1.0000 - loss: 0.0058 - val_accuracy: 1.0000 - val_loss: 0.0034
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 1.0000 - loss: 0.0027 - val_accuracy: 1.0000 - val_loss: 0.0022
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 1.0000 - loss: 0.0019 - val_accuracy: 1.0000 - val_loss: 0.0016
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 1.0000 - val_loss: 0.0012
Epoch 7/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 9.5889e-04
Epoch 8/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 1.0000 - loss: 8.6854e-04 - val_accuracy: 1.

# Explaination
1. Input sequence:-
Input text- "The refund process was fast"
After tokenization- [12, 45, 89, 10, 54]
After padding- [12, 45, 89, 10, 54, 0, 0, 0...]
2. Embedding layer:-
Embedding(input_dim=5000, output_dim=128)
Instead of 15 words become [0.12, -0.44, 0.98, ...]
3. Recurrent/sequence layer:-
LSTM(64)
The LSTM processes words sequentially and remembers important context.
4. Output layer:-
Dense(3, activation='softmax')
Since there are 3 classes, Positive, Negative, Neutral the output layer predicts class probabilities.
5. Loss function:-
sparse_categorical_crossentropy
This is suitable for multi-class classification problems. It measures prediction error during training.
6. Evaluation metric:-
Accuracy formula:
Accuracy= Total Predictions / Correct Predictions
	​
Higher accuracy means better classification performance.


# **Task 6 Attention and Transformer Reflection**

**Q.1 Why RNNs struggle with long-term dependencies?**

RNNs (Recurrent Neural Networks) process text sequentially by passing information from one time step to the next. While this helps them understand sequence data, they struggle to remember information from earlier parts of long sentences.
This happens mainly because of the vanishing gradient problem, where gradients become very small during backpropagation.

**Q.2 How LSTMs Help with Memory?**

LSTMs (Long Short-Term Memory networks) were designed to overcome the memory limitations of RNNs. LSTMs include a special memory cell and gating mechanisms that control the flow of information.

The three main gates are:

* Forget Gate
* Input Gate
* Output Gate

**Q.3 What Attention Solves in Sequence-to-Sequence Tasks?**

In traditional encoder-decoder models, the encoder compresses the entire input sequence into one fixed-size vector. This creates a bottleneck because important information may be lost when sequences become long.
The attention mechanism solves this problem by allowing the decoder to focus on the most relevant parts of the input sequence while generating each output word.

**Q. 4 Why Transformers are Important in Modern NLP and Generative AI?**

Transformers are modern deep learning architectures based primarily on the attention mechanism. Unlike RNNs and LSTMs, transformers process all words in parallel rather than sequentially.
This provides several advantages:

* Faster training
* Better scalability
* Improved contextual understanding
